### Скорости

В файле `data/sga.csv` лежит список галактик из каталога Siena Galaxy Atlas 2020 года. Каждая галактика в этом списке характеризуется именем и двумя экваториальными координатами - прямое восхождение и склонение.

<details>
<summary>Откуда данные?</summary>

Статья: https://ui.adsabs.harvard.edu/abs/2023ApJS..269....3M

Сайт обзора: https://www.legacysurvey.org/sga/sga2020/

Скачать данные можно тут (но они уже лежат у вас на компьютере): https://vizier.unistra.fr/viz-bin/VizieR?-source=J/ApJS/269/3&-to=3

</details>

В задании 1 ниже даны координаты пяти опорных точек на небе. Для каждой из них нужно найти все галактики из `sga.csv`, которые находятся ближе 10 угловых минут. Как будем действовать?

<details>
<summary>Спойлер (если ничего не придумывается)</summary>

1. Зафиксировать порог: 10 угловых минут (удобно сразу перевести в радианы).
2. Для каждой из галактик посчитать угловое расстояние большого круга до каждой галактики.
3. Для каждого объекта вывести имена всех галактик, у которых это расстояние меньше порога.

</details>

### <отступление про расстояния на сфере>

На плоскости, если известны координаты двух точек $x_1, y_1$ и $x_2$, $y_2$, расстояние между ними вычисляется из теоремы Пифагора:

$$
d = \sqrt{(x_2 - x_1)^2 + (y_2 - y_1)^2}
$$

На сфере это, строго говоря, не так:

![](images/3d_sphere_distance.png)

<details>
<summary style="color: gray;">Это, кстати, Desmos!</summary>

Позволяет рисовать произвольно сложные 3D графики: https://www.desmos.com/3d/xxphp4awjf

</details>

Расстояние на сфере между двумя точками (расстояние большого круга) считается по формуле 

$$
\theta = arccos(\sin(\delta_1)\sin(\delta_2) + \cos(\delta_1)\cos(\delta_2)\cos(\alpha_1 - \alpha_2))
$$

### </отступление про расстояния на сфере>

Попробуем для каждой из заданных опорных позиций найти близкие галактики из SGA.

<div class="alert alert-block alert-warning" style="margin-top: 20px">

<font size=4>**Задание 1**</font>     

В файле `data/sga.csv` лежит CSV-файл, в котором есть 3 столбца:
- `name` - имя галактики
- `ra` - прямое восхождение в градусах
- `dec` - склонение в градусах

Даны пять координат:

| № | RA ($\degree$) | DEC ($\degree$) |
|---|----------------|-----------------|
| 1 | 3.078541 | 31.061563 |
| 2 | 40.2805204 | -32.5505215 |
| 3 | 347.0404709 | 5.2581476 |
| 4 | 166.8269034 | 37.7940637 |
| 5 | 270.9367981 | 46.0983526 |

Для каждой из них найти и вывести имена всех галактик каталога, находящихся на расстоянии менее 10 угловых минут.

Примечание

</div>

In [ ]:
import numpy as np

targets = [
    (3.078541, 31.061563),
    (40.2805204, -32.5505215),
    (347.0404709, 5.2581476),
    (166.8269034, 37.7940637),
    (270.9367981, 46.0983526),
]
radius_rad = np.radians(10 / 60)

with open("data/sga.csv") as file:
    lines = file.readlines()

for k, (ra_g, dec_g) in enumerate(targets, start=1):
    print(f"--- Галактика {k}: RA = {ra_g}, DEC = {dec_g} ---")
    ra_g_rad = np.radians(ra_g)
    dec_g_rad = np.radians(dec_g)
    for line in lines:
        name, ra_str, dec_str = line.split(",")
        ra = float(ra_str)
        dec = float(dec_str)
        ra_rad = np.radians(ra)
        dec_rad = np.radians(dec)
        cos_theta = np.sin(dec_g_rad) * np.sin(dec_rad) + np.cos(dec_g_rad) * np.cos(dec_rad) * np.cos(ra_g_rad - ra_rad)
        theta = np.arccos(cos_theta)
        if theta < radius_rad:
            print(name, ra, dec)
    print()

### Можно ли быстрее?

В нашем примере порядок числа галактик - 100 тысяч. В современных обзорах число объектов измеряется миллионами, сотнями миллионов и миллиардами:
- SDSS-V - ~6 млн галактик
- [DESI](https://www.desi.lbl.gov/the-desi-survey/) - ~30 млн объектов
- [Gaia DR3](https://www.cosmos.esa.int/web/gaia/dr3) - ~1.46 млрд объектов

На таких масштабах и с более сложными операциями время обработки увеличивается на порядки.

Все слышали, что Python на самом деле медленный. Давайте сменим язык программирования на C и посмотрим, что будет.

<details>
<summary>Если вдруг написание кода на C не сработает</summary>

```c
#include <math.h>
#include <stdio.h>

#ifndef M_PI
#define M_PI 3.14159265358979323846
#endif

static double deg2rad(double deg) { return deg * (M_PI / 180.0); }

static double angular_separation_rad(double ra1_deg, double dec1_deg, double ra2_deg,
                                     double dec2_deg) {
    double ra1 = deg2rad(ra1_deg);
    double dec1 = deg2rad(dec1_deg);
    double ra2 = deg2rad(ra2_deg);
    double dec2 = deg2rad(dec2_deg);
    double c = sin(dec1) * sin(dec2) + cos(dec1) * cos(dec2) * cos(ra1 - ra2);
    if (c > 1.0) {
        c = 1.0;
    } else if (c < -1.0) {
        c = -1.0;
    }
    return acos(c);
}

int main(void) {
    const char *path = "data/sga.csv";
    const struct {
        double ra;
        double dec;
    } targets[] = {
        {3.078541, 31.061563},
        {40.2805204, -32.5505215},
        {347.0404709, 5.2581476},
        {166.8269034, 37.7940637},
        {270.9367981, 46.0983526},
    };
    const int n_targets = (int)(sizeof targets / sizeof targets[0]);
    const double radius_rad = deg2rad(10.0 / 60.0);

    for (int k = 0; k < n_targets; k++) {
        FILE *f = fopen(path, "r");
        if (f == NULL) {
            perror(path);
            return 1;
        }
        printf("--- Позиция %d: RA = %.7f, DEC = %.7f ---\n",
               k + 1, targets[k].ra, targets[k].dec);

        char line[512];
        char name[160];
        double ra;
        double dec;

        while (fgets(line, sizeof line, f) != NULL) {
            if (sscanf(line, "%159[^,],%lf,%lf", name, &ra, &dec) != 3) {
                continue;
            }
            if (angular_separation_rad(targets[k].ra, targets[k].dec, ra, dec) < radius_rad) {
                printf("%s %g %g\n", name, ra, dec);
            }
        }
        fclose(f);
        printf("\n");
    }
    return 0;
}
```

Компиляция:
```bash
cc -o zadanie1 zadanie1.c -lm
```

Запуск (из каталога `school/02/01_numpy`):

```bash
./zadanie1
```

</details>


### <отступление о том, как рисовать красивые изображения неба>

Для отображения изображений неба есть проект Aladin. У него несколько частей:
- Сайт для отображения карты неба (Aladin Lite): https://aladin.cds.unistra.fr/AladinLite/
- Aladin Desktop — приложение на компьютере для просмотра карт неба: https://aladin.cds.unistra.fr/java/nph-aladin.pl?frame=downloading
- Виджет для Python-ноутбуков `ipyaladin`: https://aladin.cds.unistra.fr/AladinLite/ipyaladin/

Последний нам интересен больше всего - он позволяет простым образом нарисовать нам карту неба. Представим, что мы хотим посмотреть на Крабовидную туманность:

In [ ]:
from astropy.coordinates import Angle
from ipyaladin import Aladin

aladin = Aladin(
    target="Crab",
    fov=Angle(10/60, "deg"),
)
aladin

Нужно всего два модуля:
- `astropy` - модуль с астрономическими утилитами. Подробнее мы про него поговорим на следующих школах.
- `ipyaladin` - модуль, занимающийся непосредственно рисованием.

Если хочется вместо имени объекта использовать конкретные координаты, это можно сделать вот так:

In [ ]:
from astropy.coordinates import Angle, SkyCoord
from ipyaladin import Aladin

aladin = Aladin(
    target=SkyCoord(283.396, 33.029, unit="deg"),
    fov=Angle(10/60, "deg"),
)
aladin

Этот виджет позволяет делать много чего - покликайте на разные кнопки.

### </отступление о том, как рисовать красивые изображения неба>

Результаты получили, давайте посмотрим на сами галактики (на примере одного объекта), чтобы понять, какая из них правильная:

In [ ]:
aladin = Aladin(
    target=SkyCoord(2.9988702, 30.9973874, unit="deg"),
    fov=Angle(10/60, "deg"),
)
aladin

In [ ]:
aladin = Aladin(
    target=SkyCoord(2.9887549, 30.9985883, unit="deg"),
    fov=Angle(10/60, "deg"),
)
aladin

In [ ]:
aladin = Aladin(
    target=SkyCoord(3.0785801, 31.0610539, unit="deg"),
    fov=Angle(10/60, "deg"),
)
aladin

![](images/dasha.png)

### Про скорости и удобство

На C:

```c
int read_csv(const char *filename, Galaxy galaxies[], int max_galaxies) {
    FILE *file = fopen(filename, "r");
    if (file == NULL) {
        printf("Ошибка: не удалось открыть файл %s\n", filename);
        return -1;
    }
    
    char line[MAX_LINE_LENGTH];
    int count = 0;
    
    // Если в файле есть заголовок, раскомментируйте следующую строку
    // fgets(line, MAX_LINE_LENGTH, file);
    
    while (fgets(line, MAX_LINE_LENGTH, file) != NULL && count < max_galaxies) {
        // Удаляем символ новой строки
        line[strcspn(line, "\n")] = 0;
        
        // Пропускаем пустые строки
        if (strlen(line) == 0) continue;
        
        // Парсим строку CSV
        char *token = strtok(line, ",");
        if (token == NULL) continue;
        
        // Копируем имя
        strncpy(galaxies[count].name, token, MAX_NAME_LENGTH - 1);
        galaxies[count].name[MAX_NAME_LENGTH - 1] = '\0';
        
        // Читаем прямое восхождение
        token = strtok(NULL, ",");
        if (token == NULL) continue;
        galaxies[count].ra = atof(token);
        
        // Читаем склонение
        token = strtok(NULL, ",");
        if (token == NULL) continue;
        galaxies[count].dec = atof(token);
        
        count++;
    }
    
    fclose(file);
    return count;
}
```

На Python:
```python
with open("data/sga.csv") as file:
    for line in file.readlines():
        name, ra_str, dec_str = line.split(",")
```

Вот бы был какой-то способ писать код так же удобно, как на Python, но так же быстро, как на C...

### `numpy`

Это модуль, который позволяет инструментами Python проводить манипуляции над массивами данных со скоростями, сравнимыми с низкоуровневыми языками вида C и Fortran.

Списки в Python:

![](images/pylist.png)

Массивы (векторы) `numpy` или C:

![](images/ndarray.png)

#### Пример

Задача: есть большой список чисел, нужно к каждому числу прибавить 5. Сначала сгенерируем числа:

In [ ]:
import random

numbers = []

for i in range(10000000):
    numbers.append(random.randint(1, 100))

### <отступление про измерение времени>

Для замеров времени есть модуль `time`. В частности, `time.time()` выдаёт, сколько секунд прошло с 1 января 1970 года до текущего момента.

Чтобы оценить время работы программы, достаточно вызвать эту функцию два раза:
```python
import time

start = time.time()
... #долгая операция, которую мы хотим замерить
end = time.time()

print(end - start) # в секундах
print((end - start) * 1000) # в миллисекундах
```

### </отступление про измерение времени>

И теперь посмотрим, как это сделать обычным способом:

In [ ]:
import time

num = 5
res = []

start = time.time()

for i in range(len(numbers)):
    res.append(numbers[i] + num)

end = time.time()

print((end - start) * 1000)

<details>
<summary>Что Python делает?</summary>

Для каждого элемента в списке простое, казалось бы, сложение - `lst[i] + num`, - раскрывается в целый набор операций. Для $i$-го элемента:
1. Получить, что хранится в памяти в ячейке `начало + i * 8`.
2. Перейти по полученной из этого места ссылке в другую область памяти.
3. Из этой области памяти (в ней хранится Python-объект, означающий число `61`, например) получить тип объекта - `int`.
4. Проверить по внутренней таблице - умеет ли Python складывать `int` и `int`?
5. Если да, получить алгоритм сложения, и запустить его.
6. Результат записать в память.
7. Указатель на этот участок памяти добавить в конец списка `res`.

</details>

А через `numpy`:

In [ ]:
import numpy as np
import time

lst = np.array(numbers)
num = 5

start = time.time()

res = lst + num

end = time.time()

print((end - start) * 1000)

<details>
<summary>Что numpy делает?</summary>

Один раз для всего списка:
1. Получить тип массива - `int`.
2. Получить алгоритм сложения `int` + `int`.
3. Зарезервировать место для нового массива такого же размера, как и изначальный массив.

Для каждого элемента в списке:
1. Получить, что хранится в памяти в ячейке `начало + i * 8`.
2. Выполнить алгоритм сложения результата этой ячейки с числом `5`.
3. Записать результат алгоритма в ячейку `начало + i * 8` нового массива.

</details>

## Как пользоваться `numpy`?

### Создание массива

Если имеется список данных, то из него можно создать np-вектор:

In [ ]:
import numpy as np

lst = [1, 2, 3, 4, 5]

arr = np.array(lst)

Если есть CSV-файл, из него можно также загрузить данные напрямую:

In [ ]:
arr = np.genfromtxt(
    "data/sga.csv",
    delimiter=",", # какой разделитель между столбцами?
    names=["name", "ra", "dec"], # список столбцов, если их нет в первой строке файла. Если есть, то names=True будет достаточно
    dtype=None,
)

print(arr["name"]) # получить вектор имён
print(arr["ra"]) # получить вектор прямых восхождений

Можно также создавать массивы с заранее известными значениями:

In [ ]:
print(np.zeros(500)) # массив из 500 нулей
print(np.ones(100)) # массив из 100 единиц
print(np.full(50, 77)) # массив из 50 чисел 77

Можно также генерировать наборы случайных чисел:

In [ ]:
print(np.random.uniform(0.5, 10.5, 50)) # 50 случайных чисел от 0.5 до 10.5
print(np.random.randint(10, 100, 50)) # 50 случайных целых чисел от 10 до 100

И, наконец, можно делать массивы равномерно распределённых чисел:

In [ ]:
print(np.arange(10, 100, 3)) # массив от 10 до 100 с шагом 3. То же самое, что range(10, 100, 3), но создаёт np-вектор.
print(np.linspace(10, 100, 30)) # массив из 30 равномерно распределённых точек на отрезке от 10 до 100.

### Операции над numpy-массивами

Арифметические операции с числами применяют указанную операцию к каждому элементу массива:

In [ ]:
arr = np.linspace(0, 1, 9)

print(arr + 1) # сложение с числом
print(arr - 0.1) # вычитание числа
print(arr * 1000) # умножение на число
print(2 ** arr) # возведение числа в степень

# и любые другие операции над числами, которые вам могут придуматься

У `numpy` также есть свои аналоги более сложных операций:

In [ ]:
print(np.sin(arr))
print(np.arctan(arr))
print(np.arcsin(arr) / 2 + np.arcsin(0.5) ** arr)
print(np.mod(arr * 1000, 50)) # остаток от деления на 50
print(np.max(arr)) # возвращает максимум среди элементов массива
print(np.maximum(arr, 0.5)) # возвращает максимум между каждым элементом arr и 0.5

Массивы также можно складывать (умножать, делить, etc.) друг с другом. Важная особенность - в складываемых векторах должно быть строго одинаковое число элементов.

In [ ]:
arr1 = np.array([55, 66, 77, 88, 99])
arr2 = np.array([11, 22, 11, 44, 33])

print(arr1 / arr2)

Если число элементов в массивах не совпадёт, `numpy` выдаст ошибку:

In [ ]:
arr1 = np.array([55, 66, 77, 88])
arr2 = np.array([11, 22, 11, 44, 33])

print(arr1 / arr2)

<div class="alert alert-block alert-warning" style="margin-top: 20px">

<font size=4>**Задание 2**</font>     

Сгенерировать синусоиду из 100 точек на отрезке между 0 и $\pi$.

На выходе должно быть два вектора: $x$ (от 0 до $\pi$) и $y$ (от 0 до 1 - значения синусов).

</div>

In [ ]:
import numpy as np

x = np.linspace(0, np.pi, 100)
y = np.sin(x)

<div class="alert alert-block alert-warning" style="margin-top: 20px">

<font size=4>**Задание 3**</font>     

Для каждой точки из этой синусоиды, добавить случайный равномерный шум по $y$. Шум определяется как случайное смещение от идеальной точки - размер разброса поставить 0.5.

</div>

In [ ]:
import numpy as np

x = np.linspace(0, np.pi, 100)
y = np.sin(x)
y_noisy = y + np.random.uniform(-0.5, 0.5, size=y.shape)

<div class="alert alert-block alert-warning" style="margin-top: 20px">

<font size=4>**Задание 4**</font>

Построить зашумленную синусоиду на графике.

</div>

In [ ]:
import matplotlib.pyplot as plt

plt.plot(x, y_noisy, linestyle="None", marker=".")
plt.show()